<a href="https://colab.research.google.com/github/jfodera/ai-ml-projects/blob/main/NBA_Shot_Pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [97]:
import kagglehub
import os
import glob
import pandas as pd

## Data Retrieval

### Pulling NBA Shots with Player infromation attached from existing [Kaggle Repo](https://www.kaggle.com/datasets/mexwell/nba-shots) (2004-2024)

In [98]:

# Download latest version
shotsKagglePath = kagglehub.dataset_download("mexwell/nba-shots")
shotsKagglePath = "/kaggle/input/nba-shots"

shotsCsvFiles = glob.glob(os.path.join(shotsKagglePath, "*.csv"))


Using Colab cache for faster access to the 'nba-shots' dataset.


### Retrieving Player specific Biometric Data




#### Our Process
- There were no existing Data sets that matched our needs, so we created our own by making use of the [nba_api](https://github.com/swar/nba_api)
- We created our own public [Kaggle repo](https://www.kaggle.com/datasets/joefodera/biometric-attributes-of-nba-players-per-season/data) to pull from to preserve consistency of our colab final project file
  - descriptions of how we retrieved this section of data and how we worked with the API to retrieve exactly what we needed can be found in this colab file


In [99]:

import kagglehub

# Download latest version
playerKagglePath = kagglehub.dataset_download("joefodera/biometric-attributes-of-nba-players-per-season")
print("Path to dataset files:", playerKagglePath)
playerKagglePath = "/kaggle/input/biometric-attributes-of-nba-players-per-season"
playerCsvFiles = glob.glob(os.path.join(playerKagglePath, "*.csv"))




Using Colab cache for faster access to the 'biometric-attributes-of-nba-players-per-season' dataset.
Path to dataset files: /kaggle/input/biometric-attributes-of-nba-players-per-season


### Merging and Anonymizing NBA_shots and NBA_Players


Note that based on the dataset documentation each file maps to a season with the following constructs:
- Both `NBA_2004_Shots.csv` and `NBA_2004_Players.csv` are from the 2003-2004 NBA Season



In [100]:




allPlayerDataFrames = []
for f in playerCsvFiles:
    df = pd.read_csv(f)
    allPlayerDataFrames.append(df)

allShotDataFrames = []
for f in shotsCsvFiles:
    df = pd.read_csv(f)
    allShotDataFrames.append(df)









In [101]:
#Shots Structure:
## SEASON_1,SEASON_2,TEAM_ID,TEAM_NAME,PLAYER_ID,PLAYER_NAME,POSITION_GROUP,POSITION,GAME_DATE,GAME_ID,HOME_TEAM,AWAY_TEAM,EVENT_TYPE,SHOT_MADE,ACTION_TYPE,SHOT_TYPE,BASIC_ZONE,ZONE_NAME,ZONE_ABB,ZONE_RANGE,LOC_X,LOC_Y,SHOT_DISTANCE,QUARTER,MINS_LEFT,SECS_LEFT
#Players Structure:
##PLAYER_ID,PLAYER_NAME,AGE,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,DRAFT_YEAR,SEASON_1



# Sort the dataframe lists by years so the indicies align
## .iloc[0] - looks at season from first row in DF
allShotDataFramesSort = sorted(allShotDataFrames,key=lambda x: x['SEASON_1'].iloc[0])
allPlayerDataFramesSort = sorted(allPlayerDataFrames,key=lambda x: x['SEASON_1'].iloc[0])

# # Check if indicies are aligned BEFORE dropping columns
# for i in range(len(allShotDataFramesSort)):
#   print(allShotDataFramesSort[i]['SEASON_1'].iloc[0])
#   print(allPlayerDataFramesSort[i]['SEASON_1'].iloc[0], '\n\n')

# Things to drop
dropFromPlayers = ['PLAYER_NAME']
dropFromShots = ['SEASON_1','SEASON_2','TEAM_ID','TEAM_NAME', 'PLAYER_NAME','GAME_DATE','GAME_ID','HOME_TEAM','AWAY_TEAM','QUARTER','MINS_LEFT','SECS_LEFT']
for i in range(len(allShotDataFramesSort)):
  allShotDataFramesSort[i] = allShotDataFramesSort[i].drop(columns = dropFromShots)
  allPlayerDataFramesSort[i] = allPlayerDataFramesSort[i].drop(columns = dropFromPlayers)

# # Check if correct columns have been successfully dropped
# print(allShotDataFramesSort[0].head())
# print(allPlayerDataFramesSort[0].head())


In [102]:

mergedShotBioList = []

# Since indicies are aligned, iterate through each list
for i in range(len(allShotDataFramesSort)):
  # Returns 'PLAYER_ID'
  common_cols = allShotDataFramesSort[i].columns.intersection(allPlayerDataFramesSort[i].columns).tolist()
  # "how='inner'" means only keeps rows from the cartesian product where ALL cols match from both df's
  result = pd.merge(allShotDataFramesSort[i], allPlayerDataFramesSort[i], on=common_cols, how='inner')
  result = result.drop(columns = ['PLAYER_ID'])
  mergedShotBioList.append(result)

# # Verify years for each relation are correct
# for i in range(len(mergedShotBioList)):
#   print(mergedShotBioList[i]['SEASON_1'].iloc[0])
print(mergedShotBioList[0].head()) #head of the 2004 season




# mergedShotBioList Description:
## A list of Data frames, one for each season (think of one DF as one table corresponding to an NBA Season)
## With the following Row Structured:
###[AGE,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,DRAFT_YEAR,SEASON_1,POSITION_GROUP,POSITION,EVENT_TYPE,SHOT_MADE,ACTION_TYPE,SHOT_TYPE,BASIC_ZONE,ZONE_NAME,ZONE_ABB,ZONE_RANGE,LOC_X,LOC_Y,SHOT_DISTANCE]



  POSITION_GROUP POSITION   EVENT_TYPE  SHOT_MADE         ACTION_TYPE  \
0              G       SG    Made Shot       True           Jump Shot   
1              G       PG    Made Shot       True  Driving Layup Shot   
2              G       SG  Missed Shot      False           Jump Shot   
3              G       PG    Made Shot       True           Jump Shot   
4              G       PG  Missed Shot      False           Jump Shot   

        SHOT_TYPE         BASIC_ZONE         ZONE_NAME ZONE_ABB  \
0  3PT Field Goal  Above the Break 3  Left Side Center       LC   
1  2PT Field Goal    Restricted Area            Center        C   
2  2PT Field Goal          Mid-Range  Left Side Center       LC   
3  2PT Field Goal          Mid-Range         Left Side        L   
4  2PT Field Goal          Mid-Range        Right Side        R   

        ZONE_RANGE  LOC_X  LOC_Y  SHOT_DISTANCE   AGE  PLAYER_HEIGHT_INCHES  \
0          24+ ft.   20.0  21.35             25  25.0                    78   
